In [20]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import os

from openai import OpenAI


openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [22]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

for k, v in documents[0].items():
    print(f"{k}: {v}")

id: 9e508f2212
course: data-engineering-zoomcamp
section: General Course-Related Questions
question: Course: When does the course start?
answer: A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.


In [23]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [24]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""


def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [32]:
def calculate_cost(response):
    input_price = 0.75 / 1_000_000
    output_price = 4.50 / 1_000_000

    cost = (
        response.usage.input_tokens * input_price +
        response.usage.output_tokens * output_price
    )
    return round(cost, 6)


def search(question):
    return index.search(
        question,
        boost_dict={'question': 2.0, 'section': 0.5, 'answer': 1.0},
        filter_dict={"course": "llm-zoomcamp"},
        num_results=5)


def llm(instructions, user_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini")):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response


def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    response = llm(INSTRUCTIONS, user_prompt)

    cost_in_dollars = calculate_cost(response)
    answer = response.output_text

    return (
        f"The cost to answer this question is: ${cost_in_dollars}.\n\n"
        "The answer from the llm is:\n\n"
        f"{answer}"
    )


In [41]:
question = "I just discovered the course. Can I join now?"
answer = llm(INSTRUCTIONS, question)
print(answer.output_text)

I don’t know.


In [40]:
print(rag(question))

The cost to answer this question is: $0.000429.

The answer from the llm is:

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still being accepted.
